In [ ]:
# %pip install requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# %pip install psycopg2

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import requests
import psycopg2
from datetime import datetime, timedelta
import time

# Configuração do banco
DB_CONFIG = {
    "host": "localhost",       # ou o host do container, se rodar fora do docker
    "port": 5432,
    "dbname": "dolar_db",
    "user": "postgres",
    "password": "postgres"
}

In [5]:
def consultar_cotacao_awesome():
    """
    Consulta a cotação atual do dólar (USD-BRL) via AwesomeAPI.
    Retorna compra, venda e horário da cotação.
    """
    url = "https://economia.awesomeapi.com.br/json/last/USD-BRL"
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    dados = resp.json()["USDBRL"]

    return {
        "valor_compra": float(dados["bid"]),
        "valor_venda": float(dados["ask"]),
        "data_hora": datetime.fromtimestamp(int(dados["timestamp"])),
        "fonte": "AwesomeAPI"
    }

In [6]:
def inserir_cotacao(cotacao):
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()

    query = """
        INSERT INTO cotacoes (moeda, valor_compra, valor_venda, data_hora, fonte)
        VALUES (%s, %s, %s, %s, %s)
    """
    cur.execute(query, (
        "USD",
        cotacao["valor_compra"],
        cotacao["valor_venda"],
        cotacao["data_hora"],
        cotacao["fonte"]
    ))

    conn.commit()
    cur.close()
    conn.close()
    print(f"[{datetime.now()}] Cotação inserida: compra={cotacao['valor_compra']} venda={cotacao['valor_venda']}")


In [7]:
# INTERVALO_SEGUNDOS = 5 * 60  # 5 minutos

# def monitorar():
#     while True:
#         try:
#             cotacao = consultar_cotacao_awesome()
#             inserir_cotacao(cotacao)
#         except Exception as e:
#             print(f"[{datetime.now()}] Erro: {e}")

#         time.sleep(INTERVALO_SEGUNDOS)

# monitorar()

In [9]:
cotacao_teste = consultar_cotacao_awesome()
print(cotacao_teste)
inserir_cotacao(cotacao_teste)

{'valor_compra': 5.1479, 'valor_venda': 5.1509, 'data_hora': datetime.datetime(2026, 7, 13, 21, 15, 6), 'fonte': 'AwesomeAPI'}
[2026-07-13 21:22:02.983114] Cotação inserida: compra=5.1479 venda=5.1509
